<a href="https://colab.research.google.com/github/debashisdotchatterjee/COSTAR_CVD_2026/blob/main/COSTAR_CVD_REVISED_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COSTAR-CVD V2 — complete reproducible Colab implementation

**This notebook implements the revised manuscript's conditional-gain, capacity-constrained, nominally safeguarded minimax-regret policy.** All code is embedded; no separate `.py` upload is required.

The default full execution performs nested training, exact policy optimization, held-out certification, comparator benchmarks, and **four independent feature-ablation refits**. A live overall and nested model-fitting progress bar displays elapsed and estimated remaining time. At the end a ZIP containing publication-quality figures and manually transcribable tables is automatically generated and downloaded.

Research-only offline analysis. Costs and capacity targets are hypothetical; no clinical recommendation or favorable statistical conclusion is imposed.


In [ ]:
%pip install -q "numpy>=1.24" "pandas>=2" "scipy>=1.11" "scikit-learn>=1.4" "matplotlib>=3.7" "tqdm>=4.66"

## Choose the execution profile

Set `FAST_CHECK = True` **only** for a short functionality test. Keep `FAST_CHECK = False` and `RUN_ABLATIONS = True` when generating results for the manuscript. Do not change primary constraints or loss costs based on the final holdout. The notebook asks you to upload the original CSV, then saves all outputs in a single ZIP.

In [ ]:
#!/usr/bin/env python3
"""COSTAR-CVD V2: Honest conditional-gain / safeguarded minimax-regret research pipeline.

Methodological reference: COSTAR_CVD_METHOD_REVISED/main.tex, supplied by author.
NOT clinical diagnostic software. All clinical decision costs/limits are hypothetical.
No outcome-dependent selection on the independent evaluation sample.
"""
from __future__ import annotations
import argparse, copy, hashlib, importlib.metadata, json, os, platform, sys, time, warnings, zipfile
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize
from scipy.special import expit, logit
from scipy.stats import beta
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, brier_score_loss, log_loss,
                             roc_auc_score, precision_recall_curve, roc_curve)
from sklearn.model_selection import KFold, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

B = ['age','gender','chestpain','restingBP','serumcholestrol','fastingbloodsugar','restingrelectro']
A = ['maxheartrate','exerciseangia','oldpeak','slope']
V = 'noofmajorvessels'
EXPECTED = ['patientid']+B+A+[V,'target']
CAT = {'gender','chestpain','fastingbloodsugar','restingrelectro','exerciseangia','slope',V}
BASE_NAMES = ['logistic','forest','boosting','stack']

@dataclass
class Config:
    seed: int = 20261028
    n_dev: int = 600
    n_opt: int = 200
    n_eval: int = 200
    outer_folds: int = 5  # Outer held-out fold excludes ALL tuning, stacking and calibration.
    inner_folds: int = 3
    c_test: float = .055
    c_ref: float = .18
    c_miss: float = 1.
    k_test: float = .40
    k_ref: float = .58
    mu_joint: float = .25
    nu_conditional: float = .35
    gamma: float = 1.25
    delta: float = .02
    alpha: float = .05
    bootstrap: int = 1000
    threshold_quantiles: tuple = (.10,.30,.50,.70,.85,.95)
    gain_cut_quantiles: tuple = (.05,.15,.30,.45,.60,.75,.90)
    gamma_sweep: tuple = (1.,1.25,1.5,2.)
    delta_sweep: tuple = (0.,.005,.01,.02,.04)
    fast: bool = False
    ablations: bool = True
    run_lp_checks: bool = True

class Progress:
    def __init__(self,total):
        self.bar = tqdm(total=total,desc='COSTAR overall',unit='stage',
          bar_format='{desc}: {percentage:3.0f}%|{bar:22}| {n_fmt}/{total_fmt} [{elapsed} elapsed; {remaining} ETA]')
    def step(self,message):
        self.bar.update(1)
        self.bar.set_postfix_str(message[:65]); print(f'✓ {message}',flush=True)
    def close(self): self.bar.close()


def jsave(path,obj):
    path=Path(path);path.parent.mkdir(exist_ok=True,parents=True)
    path.write_text(json.dumps(obj,indent=2,allow_nan=False,default=lambda v: float(v) if isinstance(v,np.generic) else str(v)),encoding='utf-8')

def tab(out,name,df,caption=None):
    out=Path(out)/'tables';out.mkdir(parents=True,exist_ok=True)
    df.to_csv(out/(name+'.csv'),index=False)
    # Machine-generated LaTeX is an inspection aid; manuscript tables should be transcribed and checked.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            (out/(name+'.tex')).write_text(df.to_latex(index=False,float_format=lambda v:f'{v:.4f}',
              na_rep='--',escape=True,caption=caption or name.replace('_',' ').title(),label='tab:'+name),encoding='utf-8')
        except Exception as exc: print('Table latex export warning:',exc)

def pstyle():
    plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,
      'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':280,
      'axes.grid':True,'grid.alpha':.14,'axes.axisbelow':True,'pdf.fonttype':42,
      'font.size':10,'axes.titlesize':12,'axes.labelsize':10,'legend.frameon':False})

def figsave(out,name,fig):
    f=Path(out)/'figures';f.mkdir(parents=True,exist_ok=True)
    fig.savefig(f/(name+'.png'),dpi=280,bbox_inches='tight')
    fig.savefig(f/(name+'.pdf'),bbox_inches='tight'); plt.close(fig)

def load_data(path,out):
    df=pd.read_csv(path)
    if set(df.columns)!=set(EXPECTED): raise ValueError(f'Unexpected CSV columns: {list(df.columns)}')
    if len(df)!=1000: raise ValueError('Expected the exact 1000-record Mendeley data file')
    if df.patientid.isna().any() or df.patientid.duplicated().any():raise ValueError('Patient IDs must be distinct')
    if df.target.isna().any() or not set(df.target.unique()).issubset({0,1}):raise ValueError('target must be binary 0,1')
    zero=df.serumcholestrol.eq(0)
    audit={'source_doi':'10.17632/dzz48mvjht.1','sha256':hashlib.sha256(Path(path).read_bytes()).hexdigest(),
       'n':len(df),'positive':int(df.target.sum()),'negative':int((df.target==0).sum()),
       'cholesterol_zeros':int(zero.sum()),'cholesterol_zeros_positive':int(df.loc[zero,'target'].sum()),
       'duplicate_rows':int(df.duplicated().sum()),
       'slope_vs_target':df.groupby('slope').target.agg(['size','sum']).reset_index().to_dict('records'),
       'limitations':'CSV does not verify prospective feature timing, diagnosis adjudication, patient independence, or treatment benefit'}
    jsave(Path(out)/'data_audit.json',audit)
    df=df.copy();df.loc[zero,'serumcholestrol']=np.nan
    return df,audit

def partition(df,cfg,out):
    rng=np.random.default_rng(cfg.seed)
    perm=rng.permutation(len(df));splits={'development':perm[:cfg.n_dev],
          'optimization':perm[cfg.n_dev:cfg.n_dev+cfg.n_opt],
          'evaluation':perm[cfg.n_dev+cfg.n_opt:]}
    if len(splits['evaluation'])!=cfg.n_eval:raise AssertionError('Split size mismatch')
    rows=[]
    for s,ix in splits.items():
        q=df.iloc[ix].copy();q[['patientid','target']].assign(original_row=ix).to_csv(Path(out)/(f'partition_{s}.csv'),index=False)
        rows.append({'partition':s,'n':len(ix),'positives':int(q.target.sum()),'prevalence':float(q.target.mean())})
    tab(out,'partition_summary',pd.DataFrame(rows))
    return {k:df.iloc[ix].reset_index(drop=True) for k,ix in splits.items()}

def feature_pipe(cols,indicator=True):
    cat=[c for c in cols if c in CAT]; num=[c for c in cols if c not in CAT]
    transformers=[]
    if num:
        transformers.append(('number',Pipeline([('imp',SimpleImputer(strategy='median',
          add_indicator=indicator)),('std',StandardScaler())]),num))
    if cat:
        transformers.append(('category',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
             ('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat))
    return ColumnTransformer(transformers,sparse_threshold=0)

def base_specs(cols,seed,fast=False,indicator=True):
    def pipe(clf):return Pipeline([('features',feature_pipe(cols,indicator)),('model',clf)])
    ntree=45 if fast else 105
    iters=40 if fast else 85
    return {
       'logistic':(pipe(LogisticRegression(solver='saga',penalty='elasticnet',C=1.,l1_ratio=.3,
           random_state=seed,max_iter=1800,tol=.002)),{'model__C':[1.] if fast else [.1,1.],
                         'model__l1_ratio':[.3] if fast else [.2,.7]}),
       'forest':(pipe(RandomForestClassifier(n_estimators=ntree,random_state=seed,
                  min_samples_leaf=3,n_jobs=1)),{'model__min_samples_leaf':[3] if fast else [2,6]}),
       'boosting':(pipe(HistGradientBoostingClassifier(max_iter=iters,learning_rate=.07,
                    random_state=seed,max_leaf_nodes=12)),
                    {'model__max_leaf_nodes':[10] if fast else [7,15]})}

def weights_fit(Q,y,lam):
    k=Q.shape[1]; init=np.ones(k)/k
    def f(w):
        r=Q@w-y
        return np.mean(r*r)+lam*np.sum((w-init)**2),2*Q.T@r/len(r)+2*lam*(w-init)
    sol=minimize(f,init,jac=True,method='SLSQP',bounds=[(0.,1.)]*k,
      constraints=[{'type':'eq','fun':lambda w:sum(w)-1,'jac':lambda w:np.ones(k)}],
      options={'ftol':1e-12,'maxiter':300})
    if not sol.success:raise ArithmeticError('Simplex stacking unsuccessful: '+sol.message)
    ww=np.maximum(sol.x,0);return ww/ww.sum()

def stack_lambda(Q,y,seed):
    spl=KFold(n_splits=4,shuffle=True,random_state=seed)
    ranks=[]
    for lam in [0.,.0001,.001,.01,.05]:
        vals=[]
        for a,b in spl.split(Q):
            w=weights_fit(Q[a],y[a],lam);vals.append(np.mean((Q[b]@w-y[b])**2))
        ranks.append((np.mean(vals),lam))
    return min(ranks)[1]

def cal_fit(p,y):
    x=logit(np.clip(p,1e-6,1-1e-6));y=np.asarray(y,float)
    def f(z):
        zz=z[0]+z[1]*x;res=expit(zz)-y
        return np.mean(np.logaddexp(0,zz)-y*zz), np.array([res.mean(),np.mean(res*x)])
    r=minimize(f,np.array([0.,1.]),jac=True,method='L-BFGS-B',bounds=[(None,None),(0.,30.)])
    if not r.success:raise ArithmeticError('Calibration optimization failure '+str(r.message))
    return r.x

def calibrate(raw,ab):return expit(ab[0]+ab[1]*logit(np.clip(raw,1e-6,1-1e-6)))

class RiskTier:
    """Internal OOF predictions train weights/calibration; outer held-outs are fully honest."""
    def __init__(self,cols,cfg,indicator=True,seed=0):
        self.cols=list(cols);self.cfg=cfg;self.indicator=indicator;self.seed=seed
    def fit(self,df):
        X=df[self.cols]; y=df.target.to_numpy(int); meta=np.zeros((len(y),3));self.models={};self.tunings=[]
        nfold=self.cfg.inner_folds
        for j,(name,(pipe,grid)) in enumerate(base_specs(self.cols,self.seed,self.cfg.fast,self.indicator).items()):
            cv=StratifiedKFold(n_splits=nfold,shuffle=True,random_state=self.seed+31+j)
            gs=GridSearchCV(pipe,grid,cv=cv,scoring='neg_brier_score',n_jobs=1,
                            refit=True,error_score='raise')
            gs.fit(X,y)
            best=clone(gs.best_estimator_)
            mcv=StratifiedKFold(n_splits=nfold,shuffle=True,random_state=self.seed+101+j)
            meta[:,j]=cross_val_predict(best,X,y,cv=mcv,method='predict_proba',n_jobs=1)[:,1]
            self.models[name]=gs.best_estimator_
            self.tunings.append({'tier':','.join(self.cols),'learner':name,
                                 'best_params':json.dumps(gs.best_params_),
                                 'inner_cv_brier':float(-gs.best_score_)})
        lam=stack_lambda(meta,y,self.seed+501)
        self.weights=weights_fit(meta,y,lam);self.lam=lam
        self.cal=cal_fit(meta@self.weights,y)
        self.innersample_oof=calibrate(meta@self.weights,self.cal)
        return self
    def pred(self,df):
        Q=np.column_stack([self.models[k].predict_proba(df[self.cols])[:,1]
                            for k in ['logistic','forest','boosting']])
        return {'logistic':Q[:,0],'forest':Q[:,1],'boosting':Q[:,2],
                'stack':calibrate(Q@self.weights,self.cal)}

def honest_risks(dev,cfg,progress,colsB=B,colsS=None,indicator=True,tag='primary'):
    colsS=colsS if colsS is not None else B+A
    n=len(dev); y=dev.target.to_numpy(int)
    honestB=np.full(n,np.nan);honestS=np.full(n,np.nan)
    cv=KFold(n_splits=cfg.outer_folds,shuffle=True,random_state=cfg.seed+111)
    for fold,(tr,va) in enumerate(tqdm(cv.split(dev),total=cfg.outer_folds,
             desc=f'{tag}: fully honest outer folds',leave=False)):
        trdf=dev.iloc[tr];vadf=dev.iloc[va]
        mb=RiskTier(colsB,cfg,indicator,seed=cfg.seed+2000+fold*11).fit(trdf)
        ms=RiskTier(colsS,cfg,indicator,seed=cfg.seed+3000+fold*11).fit(trdf)
        honestB[va]=mb.pred(vadf)['stack'];honestS[va]=ms.pred(vadf)['stack']
    assert np.isfinite(honestB).all() and np.isfinite(honestS).all()
    b=RiskTier(colsB,cfg,indicator,seed=cfg.seed+4100).fit(dev)
    s=RiskTier(colsS,cfg,indicator,seed=cfg.seed+5100).fit(dev)
    return b,s,honestB,honestS

class GainModel:
    def __init__(self,cols,cfg,indicator=True,seed=0):
        self.cols=cols;self.cfg=cfg;self.indicator=indicator;self.seed=seed
    def fit(self,dev,response):
        x=dev[self.cols]
        # OOF CV chooses between regularized linear and flexible nonlinear regressors.
        models={
            'ridge': Pipeline([('features',feature_pipe(self.cols,self.indicator)),('reg',Ridge(alpha=10.))]),
            'trees': Pipeline([('features',feature_pipe(self.cols,self.indicator)),
                ('reg',ExtraTreesRegressor(n_estimators=35 if self.cfg.fast else 85,
                min_samples_leaf=12,random_state=self.seed,n_jobs=1))])}
        cv=KFold(n_splits=3,shuffle=True,random_state=self.seed+51)
        scores={};oof={}
        for name,model in models.items():
            p=cross_val_predict(model,x,response,cv=cv,n_jobs=1)
            oof[name]=p;scores[name]=float(np.mean((p-response)**2))
        self.chosen=min(scores,key=lambda k:(scores[k],k))
        self.cv_mse=scores;self.dev_oof=oof[self.chosen]
        self.est=clone(models[self.chosen]).fit(x,response)
        return self
    def pred(self,df):return self.est.predict(df[self.cols])


def gain_candidates(dev,cfg,honestB,honestS,indicator=True,colsB=B):
    ys=dev.target.to_numpy(int)
    qb=np.quantile(honestB,cfg.threshold_quantiles)
    qs=np.quantile(honestS,cfg.threshold_quantiles)
    pairs=[]; logs=[]
    for ii,tb in enumerate(qb):
        for jj,ts in enumerate(qs):
            d0=(honestB>=tb).astype(int);d1=(honestS>=ts).astype(int)
            pseudo=(cfg.c_ref-cfg.c_miss*ys)*(d0-d1)-cfg.c_test
            gm=GainModel(colsB,cfg,indicator,seed=cfg.seed+6000+ii*31+jj).fit(dev,pseudo)
            cuts=np.unique(np.r_[np.quantile(gm.dev_oof,cfg.gain_cut_quantiles),0.])
            # +inf is the no-testing boundary; never use optimization scores to choose cut values.
            pairs.append({'tb':float(tb),'ts':float(ts),'gain':gm,
                          'cutoffs':list(map(float,cuts))+[float('inf')],
                          'g_id':f'gain_{ii:02d}_{jj:02d}'})
            logs.append({'pair':f'{ii}-{jj}','tb':float(tb),'ts':float(ts),
                'gain_model':gm.chosen,'ridge_cv_mse':gm.cv_mse['ridge'],
                'tree_cv_mse':gm.cv_mse['trees'],'pseudo_mean':float(np.mean(pseudo))})
    return pairs,pd.DataFrame(logs)

def development_thresholds(devscore,cfg):
    v=np.unique(np.r_[np.quantile(devscore,np.linspace(.025,.975,17)),.5,0.,1.])
    return list(map(float,v))

def make_specs(dev,cfg,pairs,bmodel):
    specs=[]
    for p in pairs:
        for tau in p['cutoffs']:
            specs.append({'kind':'gain','gid':p['g_id'],'tb':p['tb'],
                'ts':p['ts'],'tau':tau,'gain_model':p['gain'],
                'name':f"gain:{p['g_id']}:tau={tau:.6g}"})
    devpred=bmodel.pred(dev)
    for learner in BASE_NAMES:
        for threshold in development_thresholds(devpred[learner],cfg):
            specs.append({'kind':'base','name':f'{learner}:t={threshold:.6g}',
                'learner':learner,'tb':threshold})
    specs.extend([{'kind':'constant','name':'always_refer','action':1},
                  {'kind':'constant','name':'never_refer','action':0}])
    for ix,p in enumerate(specs):p['candidate_index']=ix
    return specs

def policy_decisions(specs,bpred,spred,df):
    n=len(df); T=np.zeros((len(specs),n),dtype=np.int8);R=np.zeros_like(T)
    gcache={}
    for i,s in enumerate(specs):
        if s['kind']=='gain':
            gid=s['gid']
            if gid not in gcache:gcache[gid]=s['gain_model'].pred(df)
            t=(gcache[gid]>s['tau'])
            db=(bpred['stack']>=s['tb']);ds=(spred['stack']>=s['ts'])
            T[i]=t;R[i]=np.where(t,ds,db)
        elif s['kind']=='base':R[i]=bpred[s['learner']]>=s['tb']
        else:R[i]=s['action']
    return T,R

def policy_table(T,R,y,cfg):
    y=np.asarray(y,int);n=len(y);P=y.sum()
    miss=((1-R)*y[np.newaxis,:]);L=cfg.c_test*T+cfg.c_ref*R+cfg.c_miss*miss
    rates=pd.DataFrame({'loss':L.mean(1),'testing':T.mean(1),'referral':R.mean(1),
       'joint_miss':miss.mean(1),'conditional_fn':miss.sum(1)/P if P else np.full(len(T),np.nan)})
    feas=(rates.testing<=cfg.k_test+1e-12)&(rates.referral<=cfg.k_ref+1e-12)&\
      (rates.joint_miss<=cfg.mu_joint+1e-12)&(rates.conditional_fn<=cfg.nu_conditional+1e-12)
    if P==0:feas[:]=False
    rates['feasible']=feas.to_numpy(bool)
    return rates,L

def signed_support(z,gamma):
    """Exact manuscript eq. (sortoracle), arbitrary signed last-axis contrasts."""
    z=np.asarray(z,float);n=z.shape[-1]
    if gamma<1:raise ValueError('Gamma >=1 required')
    if gamma==1:return z.mean(axis=-1)
    j=n/(gamma+1);k=int(np.floor(j));rho=j-k
    zz=np.sort(z,axis=-1)[...,::-1]
    v=(np.sum(z,axis=-1)/(gamma*n))
    extra=np.sum(zz[...,:k],axis=-1)
    if k<n:extra=extra+rho*zz[...,k]
    return v+(gamma-1/gamma)/n*extra

def oracle_check(cfg,out):
    rng=np.random.default_rng(cfg.seed+88); rec=[]
    for j in range(12):
        x=rng.normal(size=17)
        for gamma in [1.,1.25,1.5,2.]:
            N=len(x);lp=linprog(-x,A_eq=np.ones((1,N)),b_eq=[1.],
                bounds=[(1/(gamma*N),gamma/N)]*N,method='highs')
            if not lp.success:raise ArithmeticError('Oracle LP failure')
            observed=float(signed_support(x,gamma));expected=float(-lp.fun)
            if not np.isclose(observed,expected,atol=1e-9):raise AssertionError('Signed oracle incorrect')
            rec.append({'trial':j,'gamma':gamma,'sort_oracle':observed,'LP':expected,
                        'abs_error':abs(observed-expected)})
    tab(out,'signed_oracle_lp_tests',pd.DataFrame(rec))

def regret_values(losses, feasible_ids, candidate_ids, gamma, batch_size=12):
    """EXACT max over ALL feasible comparators, no regret-class pruning."""
    reference=losses[feasible_ids]
    out=np.empty(len(candidate_ids))
    for start in range(0,len(candidate_ids),batch_size):
        ix=candidate_ids[start:start+batch_size]
        diff=losses[ix,None,:]-reference[None,:,:]
        out[start:start+len(ix)]=signed_support(diff,gamma).max(axis=1)
    return out

def choose_policy(rates,L,cfg,gamma=None,delta=None,cache=None):
    gamma=cfg.gamma if gamma is None else gamma;delta=cfg.delta if delta is None else delta
    feasible=np.flatnonzero(rates.feasible.to_numpy())
    if len(feasible)==0:return {'status':'infeasible','n_feasible':0,
       'message':'NO policy satisfies all four optimization constraints'}
    loss=rates.loss.to_numpy();tr=rates.testing.to_numpy();rr=rates.referral.to_numpy()
    nom=int(sorted(feasible,key=lambda i:(loss[i],tr[i],rr[i],i))[0])
    allowed=feasible[loss[feasible]<=loss[nom]+delta+1e-12]
    # Exact pairwise oracle compares every shortlisted candidate with ALL feasible policies.
    if cache is not None and gamma in cache:
        score=cache[gamma][allowed]
    else:score=regret_values(L,feasible,allowed,gamma)
    i=int(sorted(range(len(allowed)),key=lambda j:(score[j],loss[allowed[j]],tr[allowed[j]],rr[allowed[j]],allowed[j]))[0])
    chosen=int(allowed[i]);rsel=float(score[i]);rnom=float(regret_values(L,feasible,np.array([nom]),gamma)[0])
    if rsel>rnom+1e-9 or loss[chosen]>loss[nom]+delta+1e-9:
        raise AssertionError('Safeguard theorem failed (calculation bug)')
    return {'status':'ok','primary_index':chosen,'nominal_index':nom,'n_feasible':len(feasible),
      'n_allowed':len(allowed),'regret':rsel,'nominal_regret':rnom,
      'primary_nominal_loss':float(loss[chosen]),'nominal_opt_loss':float(loss[nom]),
      'empirical_premium':float(loss[chosen]-loss[nom]),'gamma':float(gamma),'delta':float(delta)}

def cache_regret_frontier(rates,L,cfg):
    f=np.flatnonzero(rates.feasible.to_numpy()); cache={}
    for gamma in tqdm(cfg.gamma_sweep,desc='Exact signed regret frontiers',leave=False):
        cache[gamma]=np.full(len(rates),np.nan)
        if len(f):cache[gamma][f]=regret_values(L,f,f,gamma)
    return cache

def score_metrics(y,p):
    p=np.clip(p,1e-6,1-1e-6); y=np.asarray(y,int)
    x=logit(p)
    from sklearn.linear_model import LogisticRegression as LR
    if len(np.unique(y))>1:
        cl=LR(C=1e6,max_iter=1000).fit(x[:,None],y)
        slope=float(cl.coef_.ravel()[0]);intercept=float(cl.intercept_[0])
        auc=float(roc_auc_score(y,p));ap=float(average_precision_score(y,p))
    else:slope=intercept=auc=ap=np.nan
    return {'AUROC':auc,'average_precision':ap,'Brier':float(brier_score_loss(y,p)),
         'log_loss':float(log_loss(y,p,labels=[0,1])),
         'calibration_intercept':intercept,'calibration_slope':slope}

def full_metrics(y,T,R,cfg):
    y=np.asarray(y,int);T=np.asarray(T,int);R=np.asarray(R,int)
    P=int(y.sum());neg=int((1-y).sum());miss=int(np.sum(y*(1-R)))
    return {'n':len(y),'positives':P,'testing':float(np.mean(T)),
      'referral':float(np.mean(R)),'joint_miss':miss/len(y),
      'conditional_fn':miss/P if P else np.nan,'sensitivity':1-miss/P if P else np.nan,
      'specificity':float(np.sum((1-y)*(1-R)))/neg if neg else np.nan,
      'loss':float(np.mean(cfg.c_test*T+cfg.c_ref*R+cfg.c_miss*y*(1-R))),
      'miss_count':miss,'test_count':int(T.sum()),'ref_count':int(R.sum())}

def cp_up(x,n,a):
    if n<=0 or x>=n:return 1.
    return float(beta.ppf(1-a,x+1,n-x))

def cp_low(x,n,a):
    if n<=0 or x==0:return 0.
    return float(beta.ppf(a,x,n-x+1))

def certify(y,T,R,g,cfg):
    y=np.asarray(y,int);T=np.asarray(T,int);R=np.asarray(R,int);g=np.asarray(g)
    N=len(y);p=int(y.sum()); J=6;a=cfg.alpha/J
    entries=[('testing',int(T.sum()),N,cfg.k_test),
             ('referral',int(R.sum()),N,cfg.k_ref),
             ('joint_miss',int(np.sum(y*(1-R))),N,cfg.mu_joint),
             ('conditional_fn',int(np.sum(y*(1-R))),p,cfg.nu_conditional)]
    for grp in [0,1]:
        m=(g==grp)&(y==1)
        entries.append((f'conditional_fn_group_{grp}',int(np.sum((1-R)[m])),int(m.sum()),np.nan))
    return pd.DataFrame([{'endpoint':k,'count':x,'denominator':n,'point':x/n if n else np.nan,
        'familywise_upper_95':cp_up(x,n,a),'declared_limit':lim,
        'certified':bool(cp_up(x,n,a)<=lim) if np.isfinite(lim) else False,
        'alpha_endpoint':a} for k,x,n,lim in entries])

def baseline_band(dev_b,opt_b,opt_s,yopt,cfg):
    # Candidate widths fixed on development, selection on OPT; never test on evaluation.
    widths=np.unique(np.r_[0.,np.quantile(abs(dev_b-.5),[.10,.20,.35,.5,.65,.8,1.])])
    T=np.asarray([np.abs(opt_b-.5)<w for w in widths],np.int8)
    R=np.where(T,opt_s[np.newaxis,:]>=.5,opt_b[np.newaxis,:]>=.5).astype(np.int8)
    rates,L=policy_table(T,R,yopt,cfg)
    feasible=np.flatnonzero(rates.feasible.to_numpy())
    if len(feasible): ix=int(min(feasible,key=lambda j:(rates.loss.iloc[j],j)))
    else: ix=int(rates.loss.argmin())
    return {'width':float(widths[ix]),'feasible_on_opt':bool(rates.feasible.iloc[ix]),'n_feasible':len(feasible)}

def band_decision(b,s,width):
    t=(np.abs(b-.5)<width).astype(np.int8)
    r=np.where(t,s>=.5,b>=.5).astype(np.int8)
    return t,r

def random_score(ids,seed):
    # Preregistered deterministic pseudorandomization (NOT a patient-ID predictive feature).
    def one(pid):
        h=hashlib.blake2b((f'{seed}|{pid}').encode(),digest_size=8).digest()
        return int.from_bytes(h,'big')/2**64
    return np.array([one(pid) for pid in ids],float)

def bootstrap_compare(y,Te,Re,policies,cfg,out):
    names=list(policies);base='COSTAR_revised';L={}
    for k,(t,r) in policies.items():L[k]=cfg.c_test*t+cfg.c_ref*r+cfg.c_miss*y*(1-r)
    rng=np.random.default_rng(cfg.seed+871)
    Bn=cfg.bootstrap if not cfg.fast else min(200,cfg.bootstrap)
    draw=rng.integers(0,len(y),(Bn,len(y)))
    rows=[]
    for name in names:
        if name==base:continue
        d=L[base]-L[name]
        reps=d[draw].mean(1)
        rows.append({'contrast':f'COSTAR minus {name}','observed_loss_difference':d.mean(),
           'bootstrap_CI_lower':np.quantile(reps,.025),'bootstrap_CI_upper':np.quantile(reps,.975),
           'bootstrap_P_costar_less':np.mean(reps<0),'bootstrap_replications':Bn})
    df=pd.DataFrame(rows);tab(out,'paired_bootstrap_loss',df)
    return df

def score_bootstrap(y,bpred,spred,cfg,out):
    rng=np.random.default_rng(cfg.seed+942)
    bundles={'S_stack_minus_B_stack':(spred['stack'],bpred['stack']),
             'B_stack_minus_logistic':(bpred['stack'],bpred['logistic']),
             'B_stack_minus_boosting':(bpred['stack'],bpred['boosting'])}
    reps=cfg.bootstrap if not cfg.fast else min(200,cfg.bootstrap)
    draw=rng.integers(0,len(y),(reps,len(y)))
    rows=[]
    for name,(a,b) in bundles.items():
        auc=[];briers=[]
        for ids in draw:
            yy=y[ids]
            if len(np.unique(yy))==2:auc.append(roc_auc_score(yy,a[ids])-roc_auc_score(yy,b[ids]))
            briers.append(np.mean((a[ids]-yy)**2)-np.mean((b[ids]-yy)**2))
        rows.append({'contrast':name,
           'AUROC_difference':roc_auc_score(y,a)-roc_auc_score(y,b),
           'AUROC_CI_lower':np.quantile(auc,.025) if auc else np.nan,
           'AUROC_CI_upper':np.quantile(auc,.975) if auc else np.nan,
           'Brier_difference':np.mean((a-y)**2)-np.mean((b-y)**2),
           'Brier_CI_lower':np.quantile(briers,.025),
           'Brier_CI_upper':np.quantile(briers,.975),
           'bootstrap_replications':reps})
    tab(out,'paired_bootstrap_score_differences',pd.DataFrame(rows))


def plot_evaluation_regret(tab_regret,out):
    f,ax=plt.subplots(figsize=(8.1,4.8))
    ax.plot(tab_regret.gamma,tab_regret.COSTAR_revised,marker='o',lw=2,label='Revised COSTAR')
    ax.plot(tab_regret.gamma,tab_regret.Nominal_opt,marker='D',lw=2,label='Nominal optimum')
    ax.set(xlabel='Gamma (empirical bounded case-mix reweighting)',
        ylabel='Descriptive maximum pairwise regret on evaluation',
        title='Independent comparative case-mix stress test')
    ax.legend();f.tight_layout();figsave(out,'13_holdout_regret_audit',f)

def plot_risks(y,bpred,spred,out):
    fig,ax=plt.subplots(1,2,figsize=(11,4.1))
    for label,p in [('Initial stack',bpred['stack']),('Additional stack',spred['stack']),
                    ('Logistic initial',bpred['logistic'])]:
        fpr,tpr,_=roc_curve(y,p);ax[0].plot(fpr,tpr,lw=1.9,label=f'{label} (AUC {roc_auc_score(y,p):.3f})')
        precision,recall,_=precision_recall_curve(y,p)
        ax[1].plot(recall,precision,lw=1.9,label=f'{label} (AP {average_precision_score(y,p):.3f})')
    ax[0].plot([0,1],[0,1],ls=':',color='grey')
    for a,xl,yl in [(ax[0],'False-positive rate','Sensitivity'),
                    (ax[1],'Recall','Precision')]:
        a.set(xlabel=xl,ylabel=yl,xlim=(0,1),ylim=(0,1.01));a.legend(fontsize=8,loc='best')
    ax[0].set_title('ROC | independent evaluation');ax[1].set_title('Precision–recall | independent evaluation')
    fig.tight_layout();figsave(out,'01_roc_pr',fig)
    fig,ax=plt.subplots(figsize=(7.8,5.3))
    for lab,p in [('Initial stack',bpred['stack']),('Additional stack',spred['stack'])]:
        cuts=np.quantile(p,np.linspace(0,1,9));cuts=np.unique(cuts)
        buckets=pd.cut(p,cuts,include_lowest=True,duplicates='drop')
        means=pd.DataFrame({'p':p,'y':y,'q':buckets}).groupby('q',observed=True).agg(
            pred=('p','mean'),observed=('y','mean'),n=('y','size')).reset_index()
        ax.plot(means.pred,means.observed,marker='o',lw=2,label=lab)
        for row in means.itertuples():ax.annotate(str(row.n),(row.pred,row.observed),
                                                    textcoords='offset points',xytext=(0,5),fontsize=6)
    ax.plot([0,1],[0,1],ls='--',color='grey',label='Perfect calibration')
    ax.set(xlim=(0,1),ylim=(0,1),xlabel='Mean estimated risk',ylabel='Observed fraction positive',
           title='Risk calibration (numbers denote bin sample sizes)');ax.legend()
    fig.tight_layout();figsave(out,'02_calibration',fig)

def plot_policies(df,out):
    df=df.copy().sort_values('loss')
    fig,ax=plt.subplots(figsize=(max(9,1.25*len(df)),5.3))
    names=df.policy.tolist();x=np.arange(len(df))
    ax.bar(x,df.loss.values,color=['#234c6f' if a=='COSTAR_revised' else '#96b5c6' for a in names])
    ax.set(xlabel='',ylabel='Mean decision loss (hypothetical units)',title='Holdout decision loss by policy')
    ax.set_xticks(x,names,rotation=43,ha='right');fig.tight_layout();figsave(out,'03_decision_loss',fig)
    fig,ax=plt.subplots(figsize=(max(10,1.25*len(df)),5.2))
    for key in ['testing','referral','conditional_fn']:
        ax.plot(x,df[key],marker='o',label=key.replace('_',' '),lw=1.8)
    ax.axhline(.0,color='grey',lw=.5)
    ax.set(xticks=x,xticklabels=names,ylim=(0,1.02),ylabel='Proportion',
           title='Testing demand, referral demand and conditional false negatives')
    plt.setp(ax.get_xticklabels(),rotation=43,ha='right');ax.legend()
    fig.tight_layout();figsave(out,'04_policy_operating_rates',fig)

def plot_frontier(frontier,out,cfg):
    good=frontier.loc[frontier.status=='ok']
    if good.empty:return
    fig,ax=plt.subplots(1,2,figsize=(12,4.4))
    for gamma,sub in good.groupby('gamma'):
        sub=sub.sort_values('delta')
        ax[0].plot(sub.delta,sub.regret,marker='o',label=f'Γ={gamma:g}')
        ax[1].plot(sub.delta,sub.empirical_premium,marker='o',label=f'Γ={gamma:g}')
    ax[0].set(xlabel='Allowed nominal-loss premium δ',ylabel='Max comparative regret',
              title='Optimization-sample regret frontier')
    ax[1].set(xlabel='Allowed nominal-loss premium δ',ylabel='Actual nominal-loss premium',
              title='Empirical nominal safeguard')
    for a in ax:a.legend(fontsize=8)
    fig.tight_layout();figsave(out,'05_gamma_delta_frontier',fig)

def plot_certificate(cert,out):
    fig,ax=plt.subplots(figsize=(11,4.8))
    x=np.arange(len(cert)); ax.bar(x,cert.point,color='#84b0c4',label='Observed rate')
    ax.scatter(x,cert.familywise_upper_95,marker='D',color='#b04c3d',s=45,
        zorder=3,label='Exact familywise upper bound')
    for i,val in enumerate(cert.declared_limit):
        if np.isfinite(val):ax.plot([i-.35,i+.35],[val,val],lw=2,color='#444444')
    ax.set(xticks=x,xticklabels=cert.endpoint.tolist(),ylim=(0,1.08),ylabel='Fraction',
        title='Independent exact certification | dark lines: declared tolerances')
    plt.setp(ax.get_xticklabels(),rotation=24,ha='right');ax.legend(fontsize=8)
    fig.tight_layout();figsave(out,'06_exact_certification',fig)

def plot_ablation(data,out):
    if data.empty:return
    fig,ax=plt.subplots(figsize=(9,4.9));x=np.arange(len(data))
    ax.bar(x,data['COSTAR_loss'],color='#44789c',label='Revised COSTAR loss')
    ax.scatter(x,data['nominal_loss'],marker='D',color='#a45137',s=42,label='Nominal policy loss')
    ax.set(xticks=x,xticklabels=data.scenario.tolist(),ylabel='Independent evaluation loss',
          title='Leakage-sensitive feature ablations (each refitted independently)')
    plt.setp(ax.get_xticklabels(),rotation=15,ha='right');ax.legend()
    fig.tight_layout();figsave(out,'07_feature_ablation',fig)

def plot_case_mix(L,chosen,nom,gamma,out):
    d=L[chosen]-L[nom]
    n=len(d); lo=1/(gamma*n); cap=gamma/n
    w=np.full(n,lo);order=np.argsort(-d,kind='stable');extra=1-n*lo
    for idx in order:
        allocate=min(cap-lo,extra)
        if allocate<=1e-12:break
        w[idx]+=allocate;extra-=allocate
    if abs(w.sum()-1)>1e-9:raise AssertionError('Adversarial weights must sum to 1')
    if abs(w@d-signed_support(d,gamma))>1e-8:raise AssertionError('Weights not optimal')
    df=pd.DataFrame({'signed_loss_difference':d,'nominal_weight':1/n,
       'adversarial_weight':w})
    tab(out,'adversarial_weights_for_nominal_contrast',df)
    fig,ax=plt.subplots(figsize=(8.4,4.5))
    sort=np.argsort(d);ax.bar(np.arange(n),d[sort],width=1.,color=np.where(d[sort]>0,'#af5c50','#477998'))
    ax.set(xlabel='Optimization records ranked by signed cost difference',
           ylabel='COSTAR loss − nominal policy loss',
           title=f'Signed contrast emphasized by Γ={gamma:g} bounded reweighting')
    fig.tight_layout();figsave(out,'08_signed_regret_contrast',fig)

def plot_gain(pairs,dev,opt,spec,selected_idx,out):
    s=spec[selected_idx]
    if s['kind']!='gain':return
    g=s['gain_model'].pred(opt);cut=s['tau']
    fig,ax=plt.subplots(figsize=(8.4,4.7))
    ax.hist(g,bins=26,alpha=.75,color='#4b84a2',edgecolor='white')
    if np.isfinite(cut):ax.axvline(cut,color='#ba593f',label=f'Chosen cutoff {cut:.3f}',lw=2)
    ax.axvline(0,color='#2b403c',ls='--',lw=1.8,label='Zero net gain')
    ax.set(xlabel='Predicted conditional diagnostic gain',ylabel='Optimization records',
           title='Selective acquisition score (trained only on development)')
    ax.legend();fig.tight_layout();figsave(out,'09_conditional_gain_distribution',fig)

def plot_group(y,T,R,g,out):
    rows=[]
    for v in [0,1]:
        idx=(g==v);yy=y[idx];pp=yy.sum()
        rows.append({'group':f'gender code {v}','n':int(idx.sum()),'positives':int(pp),
            'testing':float(np.mean(T[idx])) if idx.sum() else np.nan,
            'referral':float(np.mean(R[idx])) if idx.sum() else np.nan,
            'false_negative':float(np.sum(yy*(1-R[idx]))/pp) if pp else np.nan})
    d=pd.DataFrame(rows);fig,ax=plt.subplots(figsize=(8,4.4))
    xx=np.arange(len(d));w=.25
    for j,name in enumerate(['testing','referral','false_negative']):
        ax.bar(xx+(j-1)*w,d[name],w,label=name)
    ax.set(xticks=xx,xticklabels=d.group,ylim=(0,1),ylabel='Rate',
           title='Subgroup diagnostic rates with denominator-sensitive interpretation');ax.legend()
    fig.tight_layout();figsave(out,'10_subgroup_diagnostics',fig)
    return d

def plot_capacity_frontier(front,out):
    q=front.loc[front.status=='ok'].sort_values('referral_capacity')
    if q.empty:return
    fig,ax=plt.subplots(1,2,figsize=(11.6,4.4))
    ax[0].plot(q.referral_capacity,q.independent_COSTAR_loss,marker='o',label='COSTAR')
    ax[0].plot(q.referral_capacity,q.independent_nominal_loss,marker='D',label='Nominal')
    ax[0].set(xlabel='Declared referral capacity',ylabel='Independent evaluation loss',
       title='Exploratory capacity frontier (independently assessed)')
    ax[1].plot(q.referral_capacity,q.independent_testing,marker='o',label='Testing')
    ax[1].plot(q.referral_capacity,q.independent_referral,marker='D',label='Referral')
    ax[1].set(xlabel='Declared referral capacity',ylabel='Holdout operational rate',
      title='Capacity usage from policies chosen on optimization sample')
    for a in ax:a.legend()
    fig.tight_layout();figsave(out,'12_resource_capacity_frontier',fig)

def plot_cost_sensitivity(policies,y,cfg,out):
    rows=[]
    for cR in [.08,.12,.18,.25,.35]:
        for cT in [.025,.055,.09]:
            for name,(t,r) in policies.items():
                rows.append({'referral_cost':cR,'testing_cost':cT,'policy':name,
                 'fixed_policy_loss':float(np.mean(cT*t+cR*r+cfg.c_miss*y*(1-r)))})
    data=pd.DataFrame(rows)
    tab(out,'fixed_policy_cost_sensitivity',data)
    fig,ax=plt.subplots(figsize=(9.0,5.1))
    for name in ['COSTAR_revised','Nominal_opt','Logistic_opt']:
        if name not in policies:continue
        q=data[(data.policy==name)&(np.isclose(data.testing_cost,cfg.c_test))]
        ax.plot(q.referral_cost,q.fixed_policy_loss,marker='o',label=name)
    ax.set(xlabel='Hypothetical relative referral cost',ylabel='Independent-sample loss',
           title='Cost sensitivity of FROZEN evaluation policies');ax.legend()
    fig.tight_layout();figsave(out,'11_fixed_policy_cost_sensitivity',fig)

def generalization_bound(K,n,M,eta):
    return float(M*np.sqrt(2*np.log(2*K*K/eta)/n))

def run_ablation(name,dev,opt,ev,cfg,colsB,colsS,indicator,out):
    # Each independent ablation uses original labels only in development and optimization,
    # independently selecting its own policy BEFORE evaluating on ev.
    b,s,ob,os=honest_risks(dev,cfg,None,colsB,colsS,indicator,tag=name)
    pairs,gl=gain_candidates(dev,cfg,ob,os,indicator,colsB)
    specs=make_specs(dev,cfg,pairs,b)
    T,R=policy_decisions(specs,b.pred(opt),s.pred(opt),opt)
    rate,L=policy_table(T,R,opt.target.to_numpy(int),cfg)
    choice=choose_policy(rate,L,cfg)
    if choice['status']!='ok':
        return {'scenario':name,'status':'infeasible','n_feasible':0,
                'COSTAR_loss':np.nan,'nominal_loss':np.nan,'COSTAR_testing':np.nan}
    evT,evR=policy_decisions([specs[choice['primary_index']],specs[choice['nominal_index']]],
                             b.pred(ev),s.pred(ev),ev)
    me=full_metrics(ev.target.to_numpy(int),evT[0],evR[0],cfg)
    mn=full_metrics(ev.target.to_numpy(int),evT[1],evR[1],cfg)
    return {'scenario':name,'status':'ok','n_feasible':choice['n_feasible'],
            'COSTAR_loss':me['loss'],'nominal_loss':mn['loss'],
            'COSTAR_testing':me['testing'],'COSTAR_joint_miss':me['joint_miss'],
            'COSTAR_conditional_fn':me['conditional_fn'],
            'nominal_premium_on_optimization':choice['empirical_premium'],
            'COSTAR_empirical_regret':choice['regret']}

def report(out,cfg,audit,sel,metrics,scores,cert,front,abl):
    p=Path(out)/'README_RESULTS.md'
    def get(name,col):
        z=metrics.loc[metrics.policy==name]
        return 'not available' if z.empty else f'{z.iloc[0][col]:.4f}'
    f=[f'# COSTAR-CVD revised — execution report','',
       'All decision costs and capacity limits are hypothetical scenario parameters, not empirically calibrated clinical costs or established safety targets.','',
       '## Primary policy (frozen before evaluation)','',
       f'- Optimization status: **{sel["status"]}**',
       f'- Gamma: {cfg.gamma}; allowed nominal loss premium delta: {cfg.delta}',
       f'- Feasible optimization candidates: {sel.get("n_feasible",0)}',
       f'- Exact maximum pairwise regret on optimization sample: {sel.get("regret",float("nan")):.6f}',
       f'- Nominal-optimization comparator regret: {sel.get("nominal_regret",float("nan")):.6f}',
       f'- Optimization nominal loss premium: {sel.get("empirical_premium",float("nan")):.6f}',
       '', '## Independent evaluation',
       f'- Revised COSTAR decision loss: {get("COSTAR_revised","loss")}',
       f'- Nominal-optimal decision loss: {get("Nominal_opt","loss")}',
       f'- Logistic-optimized decision loss: {get("Logistic_opt","loss")}',
       f'- Revised COSTAR testing rate: {get("COSTAR_revised","testing")}',
       f'- Revised COSTAR sensitivity: {get("COSTAR_revised","sensitivity")}',
       '', '## Scientific interpretation',
       '- The two safeguard inequalities are properties of the **optimization sample**; they are not a guarantee of holdout superiority.',
       '- Pairwise policy comparisons use the same independent evaluation records and paired bootstrap confidence intervals.',
       '- Any baseline marked infeasible must not be advertised as satisfying the same resource limits.',
       '- Independent exact certificates apply to a fixed selected policy under the independent and identically distributed assumption.',
       '- All reported post-selection or bootstrap summaries of additional tuning scenarios are descriptive unless multiplicity is handled.',
       '- The 600/200/200 split is a one-time random partition, not a proof of patient independence or prospective validation.',
       '- The test benefit is a hypothetical retrospective label-based loss improvement, not a clinical causal effect.',
       '- Synthetic cost parameters can alter ranks: reported benefit must be explicitly conditional on them.',
       '- Feature timing and meaning of the recorded target remain unvalidated.',
       '', '## Machine-readable output','',
       'The `tables` directory contains CSV and LaTeX exports; `figures` contains matched PNG and vector PDF plots.',
       'The `predictions` directory contains opt/eval diagnostics (never used to retune the primary policy).',
       'The `manifest.json` records exact package versions, seeds, configuration, and source checksum.',
       'All declared uncertainty/robustness scenarios and all infeasible outcomes remain available for inspection.',
       '', f'## Uniform finite-class bound',
       f'The conservative excess-risk contrast radius is {generalization_bound(sel.get("K",1),cfg.n_opt,cfg.c_test+max(cfg.c_ref,cfg.c_miss),.05):.4f} (loss units).',
       'Its possible vacuity on n=200 is an expected limitation, not a missing computation.']
    p.write_text('\n'.join(f)+'\n',encoding='utf-8')

def zip_folder(out,zipdest):
    out=Path(out);zipdest=Path(zipdest)
    with zipfile.ZipFile(zipdest,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,arcname=str(Path('COSTAR_CVD_REVISED_RESULTS')/p.relative_to(out)))
    print(f'FINAL ZIP: {zipdest.resolve()} | {zipdest.stat().st_size/1024**2:.2f} MB',flush=True)
    try:
        from google.colab import files
        files.download(str(zipdest))
    except ImportError:
        print('Not in Google Colab; the ZIP remains at the printed path.')


def locate_dataset(path):
    if path and Path(path).exists():return Path(path)
    for p in [Path('Cardiovascular_Disease_Dataset.csv'),Path('/content/Cardiovascular_Disease_Dataset.csv'),
              Path('/mnt/data/Cardiovascular_Disease_Dataset.csv')]:
        if p.exists():return p
    try:
        from google.colab import files
        print('Upload the original Cardiovascular_Disease_Dataset.csv file.')
        uploaded=files.upload()
        for filename in uploaded:
            if filename.lower().endswith('.csv'):return Path(filename)
    except ImportError:pass
    raise FileNotFoundError('Supply --data /path/to/Cardiovascular_Disease_Dataset.csv')


def run(cfg,data,out,zipdest):
    start=time.time();out=Path(out);out.mkdir(parents=True,exist_ok=True)
    for p in ['figures','tables','predictions','verification']: (out/p).mkdir(exist_ok=True)
    pstyle()
    prog=Progress(15+(4 if cfg.ablations else 0))
    try:
        df,audit=load_data(data,out);split=partition(df,cfg,out)
        dev,opt,ev=split['development'],split['optimization'],split['evaluation']
        prog.step('Dataset schema, data audit and unstratified independent partitions')
        if cfg.run_lp_checks:oracle_check(cfg,out)
        prog.step('Signed reweighting oracle verified against SciPy LP')
        b,s,hb,hs=honest_risks(dev,cfg,prog)
        pd.DataFrame({'patientid':dev.patientid,'y':dev.target,'honest_B':hb,'honest_S':hs}).to_csv(
            out/'predictions'/'development_honest_risks.csv',index=False)
        jsave(out/'risk_fit_config.json',{'B_stack_weights':list(map(float,b.weights)),
            'S_stack_weights':list(map(float,s.weights)),
            'B_calibration':list(map(float,b.cal)),'S_calibration':list(map(float,s.cal)),
            'B_tuning':b.tunings,'S_tuning':s.tunings})
        prog.step('Nested cross-fitted honest risks and final B/S stacked calibrations')
        pairs,gain_log=gain_candidates(dev,cfg,hb,hs)
        tab(out,'gain_model_cv',gain_log)
        prog.step('36 conditional-gain regressions fitted/tuned using honest pseudo-responses')
        specs=make_specs(dev,cfg,pairs,b)
        ob,os=b.pred(opt),s.pred(opt)
        to,ro=policy_decisions(specs,ob,os,opt)
        rates,L=policy_table(to,ro,opt.target.to_numpy(int),cfg)
        rates['index']=np.arange(len(rates)); rates['policy']=[s['name'] for s in specs]
        rates['kind']=[s['kind'] for s in specs]
        tab(out,'entire_policy_universe_optimization',rates)
        prog.step(f'Frozen finite policy universe evaluated (K={len(specs)})')
        cache=cache_regret_frontier(rates,L,cfg)
        primary=choose_policy(rates,L,cfg,cache=cache)
        if primary['status']!='ok':
            tab(out,'candidate_infeasibility',rates)
            jsave(out/'infeasibility_notice.json',{'status':'infeasible','constraints':asdict(cfg),
               'message':'No candidate satisfies all four constraints; do not relax using evaluation outcomes.'})
            zip_folder(out,zipdest)
            print('INFEASIBLE: Diagnostic ZIP saved without using evaluation labels.')
            return out
        primary['K']=len(specs)
        jsave(out/'primary_selection.json', {**primary,
           'selected_policy':specs[primary['primary_index']]['name'],
           'nominal_policy':specs[primary['nominal_index']]['name']})
        f=np.flatnonzero(rates.feasible.to_numpy())
        # Full exact regret for each feasible policy at designated gamma.
        rvec=cache[cfg.gamma] if cfg.gamma in cache else None
        if rvec is None:
            rvec=np.full(len(specs),np.nan);rvec[f]=regret_values(L,f,f,cfg.gamma)
        rates['primary_gamma_regret']=rvec
        tab(out,'complete_feasible_regret_optimization',rates)
        prog.step('Full-class exact signed minimax regret + nominal safeguard certified')
        fronts=[]
        for gamma in cfg.gamma_sweep:
            for delta in cfg.delta_sweep:
                ans=choose_policy(rates,L,cfg,gamma,delta,cache=cache)
                fronts.append({'gamma':gamma,'delta':delta,**ans})
        frontier=pd.DataFrame(fronts);tab(out,'gamma_delta_frontier',frontier)
        plot_frontier(frontier,out,cfg)
        prog.step('Full Gamma×delta frontier, independent of evaluation outcomes')
        # Baseline policy selection on optimization ONLY.
        baseline_ids={}
        for learner in BASE_NAMES:
            ids=[j for j,p in enumerate(specs) if p['kind']=='base' and p['learner']==learner]
            feasible_ids=[i for i in ids if rates.feasible.iloc[i]]
            if feasible_ids:baseline_ids[learner]=int(min(feasible_ids,key=lambda i:(rates.loss.iloc[i],i)))
        logistic_fixed={'kind':'manual','name':'Logistic_threshold_0.5'}
        band=baseline_band(b.pred(dev)['stack'],ob['stack'],os['stack'],opt.target.to_numpy(int),cfg)
        prog.step('Basic optimized classifier policies and middle-band comparator frozen')
        y=ev.target.to_numpy(int);eb,es=b.pred(ev),s.pred(ev)
        names=['COSTAR_revised','Nominal_opt']+[k.title()+'_opt' for k in baseline_ids]
        candidate_ids=[primary['primary_index'],primary['nominal_index']]+[baseline_ids[k] for k in baseline_ids]
        te,re=policy_decisions([specs[i] for i in candidate_ids],eb,es,ev)
        policies={n:(te[i].astype(int),re[i].astype(int)) for i,n in enumerate(names)}
        policies['Logistic_fixed_0.5']=(np.zeros(len(y),dtype=int),(eb['logistic']>=.5).astype(int))
        policies['Middle_risk_band']=band_decision(eb['stack'],es['stack'],band['width'])
        # Matched-rate random testing, opt-only threshold selection.
        p_random=float(rates.testing.iloc[primary['primary_index']])
        uro=random_score(opt.patientid,cfg.seed+322)
        train=(uro<p_random).astype(int)
        possible=development_thresholds(b.pred(dev)['stack'],cfg)
        vchoice=[]
        for t0 in possible:
            for t1 in possible:
                rox=np.where(train,os['stack']>=t1,ob['stack']>=t0).astype(int)
                rt,Lt=policy_table(train[None,:],rox[None,:],opt.target.to_numpy(int),cfg)
                if bool(rt.feasible.iloc[0]):vchoice.append((float(rt.loss.iloc[0]),t0,t1))
        if vchoice:
            _,t0,t1=min(vchoice);ure=random_score(ev.patientid,cfg.seed+322)
            tr=(ure<p_random).astype(int)
            rr=np.where(tr,es['stack']>=t1,eb['stack']>=t0).astype(int)
            policies['Matched_random_panel']=(tr,rr)
        primary_t,primary_r=policies['COSTAR_revised']
        optchoice={'selected_kind':specs[primary['primary_index']]['kind'],
             'selected_thresholds': {k:specs[primary['primary_index']].get(k) for k in ['tb','ts','tau']},
             'middle_band_baseline':band,'matched_random_included':bool(vchoice)}
        jsave(out/'policy_metadata.json',optchoice)
        met=[];pr=[]
        opt_feasible={int(i):bool(rates.feasible.iloc[i]) for i in candidate_ids}
        for n,(t,r) in policies.items():
            row=full_metrics(y,t,r,cfg);row['policy']=n
            # On holdout, this means observed satisfaction, NOT post-hoc certification.
            row['observed_evaluation_constraints_met']=bool(
                row['testing']<=cfg.k_test+1e-12 and row['referral']<=cfg.k_ref+1e-12 and
                row['joint_miss']<=cfg.mu_joint+1e-12 and row['conditional_fn']<=cfg.nu_conditional+1e-12)
            met.append(row)
            for j in range(len(y)):
                pr.append({'patientid':ev.patientid.iloc[j],'y':int(y[j]),'policy':n,
                  'test':int(t[j]),'refer':int(r[j]),'loss':float(cfg.c_test*t[j]+cfg.c_ref*r[j]+cfg.c_miss*y[j]*(1-r[j]))})
        metrics=pd.DataFrame(met);tab(out,'independent_policy_metrics',metrics)
        pd.DataFrame(pr).to_csv(out/'predictions'/'independent_policy_decisions_long.csv',index=False)
        pd.DataFrame({'patientid':ev.patientid,'y':y,
          **{('pB_'+k):v for k,v in eb.items()},
          **{('pS_'+k):v for k,v in es.items()}}).to_csv(
            out/'predictions'/'independent_score_predictions.csv',index=False)
        scoretab=[]
        for stage,pred in [('initial',eb),('additional',es)]:
            for k,val in pred.items():scoretab.append({'stage':stage,'learner':k,**score_metrics(y,val)})
        # Deliberately excluded downstream field: retrospective score-only stress test.
        # Its predictions NEVER enter the primary gain regressions or policy optimization.
        vm=RiskTier(B+A+[V],cfg,indicator=True,seed=cfg.seed+7700).fit(dev)
        vp=vm.pred(ev)['stack']
        scoretab.append({'stage':'downstream_V_retrospective_ONLY','learner':'stack',
                         **score_metrics(y,vp)})
        downstream=pd.DataFrame({'patientid':ev.patientid,'y':y,'retrospective_pBAV':vp})
        downstream.to_csv(out/'predictions'/'retrospective_downstream_variable_scores.csv',index=False)
        scores=pd.DataFrame(scoretab);tab(out,'independent_score_metrics',scores)
        score_bootstrap(y,eb,es,cfg,out)
        plot_risks(y,eb,es,out);plot_policies(metrics,out)
        fig,ax=plt.subplots(figsize=(7.3,4.6))
        for label,pred in [('B, deployability scenario',eb['stack']),
                           ('B+A, additional panel',es['stack']),
                           ('B+A+V, retrospective ONLY',vp)]:
            fpr,tpr,_=roc_curve(y,pred)
            ax.plot(fpr,tpr,label=f'{label}: AUC={roc_auc_score(y,pred):.3f}',lw=1.8)
        ax.plot([0,1],[0,1],ls=':',color='grey')
        ax.set(xlabel='False-positive rate',ylabel='Sensitivity',
               title='Downstream-field sensitivity (not available to screening policy)')
        ax.legend(fontsize=8)
        fig.tight_layout();figsave(out,'14_downstream_variable_retrospective',fig)
        prog.step('Frozen policies evaluated on independent holdout; score and decision tables')
        cert=certify(y,primary_t,primary_r,ev.gender.to_numpy(),cfg)
        tab(out,'independent_exact_certification',cert);plot_certificate(cert,out)
        margin=generalization_bound(len(specs),cfg.n_opt,cfg.c_test+max(cfg.c_ref,cfg.c_miss),cfg.alpha)
        jsave(out/'formal_verification.json',{'nominal_optimization_safeguard':
            primary['empirical_premium']<=cfg.delta+1e-10,
          'comparative_regret_safeguard':primary['regret']<=primary['nominal_regret']+1e-10,
          'finite_class_bound':margin,'bound_eta':cfg.alpha,
          'nominal_population_excess_loss_upper':cfg.delta+margin,
          'exact_familywise_certification_all_four':bool(cert.certified.iloc[:4].all()),
          'all_four_certification_description':'Only true when all four predeclared CP upper limits meet their predeclared thresholds'})
        prog.step('Exact six-endpoint post-selection binomial risk certificates')
        bootstrap_compare(y,primary_t,primary_r,policies,cfg,out)
        plot_case_mix(L,primary['primary_index'],primary['nominal_index'],cfg.gamma,out)
        plot_gain(pairs,dev,opt,specs,primary['primary_index'],out)
        gr=plot_group(y,primary_t,primary_r,ev.gender.to_numpy(),out)
        tab(out,'evaluation_gender_diagnostics',gr)
        prog.step('Paired bootstrap, adversarial case-mix check, gain and subgroup diagnostics')
        plot_cost_sensitivity(policies,y,cfg,out)
        prog.step('Frozen-policy cost-ratio sensitivity (not relabeled as refitted experiments)')
        # Independent diagnostic regret: original opt-feasible comparator set retained.
        Te_all,Re_all=policy_decisions(specs,eb,es,ev)
        eval_rates,Le_all=policy_table(Te_all,Re_all,y,cfg)
        independent_regret=[]
        for gamma in cfg.gamma_sweep:
            ids=np.array([primary['primary_index'],primary['nominal_index']])
            vals=regret_values(Le_all,f,ids,gamma)
            independent_regret.append({'gamma':gamma,'COSTAR_revised':vals[0],
                'Nominal_opt':vals[1],'difference_C_minus_N':vals[0]-vals[1],
                'comparison_class':'optimization-feasible policies, frozen before holdout'})
        independent_regret=pd.DataFrame(independent_regret)
        tab(out,'independent_descriptive_regret',independent_regret)
        plot_evaluation_regret(independent_regret,out)
        capfront=[]
        for cap in [.35,.40,.45,.50,.55,.58,.65]:
            sub=replace(cfg,k_ref=cap)
            shifted=rates.copy()
            shifted['feasible']=(shifted.testing<=sub.k_test+1e-12)&(shifted.referral<=sub.k_ref+1e-12)&\
                (shifted.joint_miss<=sub.mu_joint+1e-12)&(shifted.conditional_fn<=sub.nu_conditional+1e-12)
            cs=choose_policy(shifted,L,sub)
            entry={'referral_capacity':cap,**cs}
            if cs['status']=='ok':
                ii=cs['primary_index'];nn=cs['nominal_index']
                em=full_metrics(y,Te_all[ii],Re_all[ii],cfg)
                en=full_metrics(y,Te_all[nn],Re_all[nn],cfg)
                entry.update({'independent_COSTAR_loss':em['loss'],
                    'independent_nominal_loss':en['loss'],
                    'independent_testing':em['testing'],
                    'independent_referral':em['referral'],
                    'independent_sensitivity':em['sensitivity'],
                    'selected_policy_kind':specs[ii]['kind']})
            capfront.append(entry)
        capfront=pd.DataFrame(capfront)
        tab(out,'capacity_frontier_exploratory',capfront)
        plot_capacity_frontier(capfront,out)
        prog.step('Independent fixed-class regret audit and optimization-only capacity frontiers')
        # Repeated fitting ablations are exploratory descriptive follow-ups; evaluation
        # draws are shared, but no ablation may select/tune the primary policy.
        ablations=[]
        if cfg.ablations:
            ablation_specs=[('without_slope',B,B+[x for x in A if x!='slope'],True),
                 ('without_slope_oldpeak',B,B+[x for x in A if x not in ('slope','oldpeak')],True),
                 ('without_cholesterol_indicator',B,B+A,False),
                 ('without_cholesterol_variable',[c for c in B if c!='serumcholestrol'],
                    [c for c in B if c!='serumcholestrol']+A,False)]
            for title,cb,cs,indic in ablation_specs:
                row=run_ablation(title,dev,opt,ev,cfg,cb,cs,indic,out)
                ablations.append(row);prog.step('Independent leakage ablation: '+title)
        ablations.insert(0,{'scenario':'primary','status':'ok',
             'n_feasible':primary['n_feasible'],'COSTAR_loss':full_metrics(y,primary_t,primary_r,cfg)['loss'],
             'nominal_loss':float(metrics.loc[metrics.policy=='Nominal_opt','loss'].iloc[0]),
             'COSTAR_testing':float(primary_t.mean())})
        ablat=pd.DataFrame(ablations)
        tab(out,'feature_ablation_holdout',ablat)
        plot_ablation(ablat.loc[ablat.status=='ok'],out)
        # Replay all held-out data checks but no post-hoc adjustment of primary.
        log={'started_utc':datetime.now(timezone.utc).isoformat(),
             'dataset_sha256':audit['sha256'],'config':asdict(cfg),'seed':cfg.seed,
             'versions':{n:importlib.metadata.version(n) for n in
                ['numpy','pandas','scipy','scikit-learn','matplotlib','tqdm']},
             'python':platform.python_version(),
             'analysis_endpoints':'independent evaluation only, selected policy frozen',
             'elapsed_seconds':time.time()-start}
        jsave(out/'manifest.json',log)
        report(out,cfg,audit,primary,metrics,scores,cert,frontier,ablat)
        prog.step('Provenance manifest, interpretation report and final integrity checks')
        # 14 + 3 = 17 stages; account precisely and package separately.
        prog.step('High-resolution PNG, vector PDF, CSV and LaTeX exports finalized')
    finally:prog.close()
    zip_folder(out,zipdest)
    return out

def unit_tests():
    # Tests for exact oracle including signed contrasts, feasible regret representation,
    # gain sign, empirical safeguard, and independent certification handling.
    x=np.array([-3.,-1.,2.,4.,7.]);n=len(x)
    for gamma in [1,1.05,1.25,1.5,2.,3.]:
        lp=linprog(-x,A_eq=np.ones((1,n)),b_eq=[1.],bounds=[(1/(gamma*n),gamma/n)]*n)
        assert lp.success and abs(signed_support(x,gamma)+lp.fun)<1e-10
    rng=np.random.default_rng(4);loss=rng.random((5,23))
    for gamma in [1,1.4,2]:
        w=np.ones(23)/23
        qreg=regret_values(loss,np.arange(5),np.arange(5),gamma)
        assert np.all(qreg>=-1e-8)
        assert np.all(qreg>=np.mean(loss,axis=1)-min(np.mean(loss,axis=1))-1e-8)
        if gamma==1:assert np.allclose(qreg,loss.mean(1)-min(loss.mean(1)))
    c=Config();T=np.zeros((3,20));R=np.zeros((3,20));R[1]=1;T[2]=1;R[2]=1
    y=np.r_[np.ones(10),np.zeros(10)]
    rates,LL=policy_table(T,R,y,c)
    assert 'conditional_fn' in rates and np.isclose(rates.conditional_fn.iloc[1],0)
    assert np.isclose(LL[2].mean(),c.c_test+c.c_ref)
    # Regression test for pseudo-gain equation (corresponds to L0-L1).
    d0=np.array([0,1,1]);d1=np.array([1,0,1]);yy=np.array([1,1,0]);
    observed=(c.c_ref-c.c_miss*yy)*(d0-d1)-c.c_test
    L0=c.c_ref*d0+c.c_miss*yy*(1-d0)
    L1=c.c_test+c.c_ref*d1+c.c_miss*yy*(1-d1)
    assert np.allclose(observed,L0-L1)
    assert cp_up(0,0,.01)==1 and cp_up(3,3,.01)==1
    print('All revised mathematics and decision-definition unit tests PASSED')


def main(argv=None):
    a=argparse.ArgumentParser(description='COSTAR V2: development-only selection, independent evaluation, ZIP export')
    a.add_argument('--data',default=None)
    a.add_argument('--out',default='COSTAR_CVD_REVISED_RESULTS')
    a.add_argument('--zip',default='COSTAR_CVD_REVISED_RESULTS.zip')
    a.add_argument('--fast',action='store_true',help='Reduced trees, CV grids and bootstrap (testing only)')
    a.add_argument('--no-ablations',action='store_true')
    a.add_argument('--tests-only',action='store_true')
    a.add_argument('--delta',type=float,default=.02)
    a.add_argument('--gamma',type=float,default=1.25)
    a.add_argument('--bootstrap',type=int,default=1000)
    a.add_argument('--k-test',type=float,default=.4)
    a.add_argument('--k-ref',type=float,default=.58)
    a.add_argument('--mu-joint',type=float,default=.25)
    a.add_argument('--nu-conditional',type=float,default=.35)
    a.add_argument('--c-test',type=float,default=.055)
    a.add_argument('--c-ref',type=float,default=.18)
    a.add_argument('--c-miss',type=float,default=1.)
    args=a.parse_args(argv)
    unit_tests()
    if args.tests_only:return
    cfg=Config(fast=args.fast,ablations=not args.no_ablations,delta=args.delta,
        gamma=args.gamma,bootstrap=args.bootstrap,k_test=args.k_test,k_ref=args.k_ref,
        mu_joint=args.mu_joint,nu_conditional=args.nu_conditional,
        c_test=args.c_test,c_ref=args.c_ref,c_miss=args.c_miss)
    if min(cfg.c_test,cfg.c_ref)<0 or cfg.c_miss<=0:raise ValueError('Costs require nonnegative test/referral and positive miss cost')
    if cfg.gamma not in cfg.gamma_sweep:cfg.gamma_sweep=tuple(sorted(set(cfg.gamma_sweep+(cfg.gamma,))))
    if cfg.delta not in cfg.delta_sweep:cfg.delta_sweep=tuple(sorted(set(cfg.delta_sweep+(cfg.delta,))))
    if cfg.bootstrap<20:raise ValueError('Use at least 20 bootstrap replications')
    source=locate_dataset(args.data)
    run(cfg,source,args.out,args.zip)




# ============================ COLAB RUN CONFIGURATION ============================
FAST_CHECK = False         # True ONLY for short functionality checks
RUN_ABLATIONS = True       # Four independent feature ablation refits
N_BOOTSTRAP = 1500         # Manuscript-grade paired bootstrap replicates

unit_tests()
config = Config(fast=FAST_CHECK, ablations=RUN_ABLATIONS,
                bootstrap=N_BOOTSTRAP)
original_csv = locate_dataset(None)
results_folder = run(config, original_csv,
                     out='COSTAR_CVD_REVISED_RESULTS',
                     zipdest='COSTAR_CVD_REVISED_RESULTS.zip')
print('\nOutput folder:', results_folder)
print('FINAL downloadable ZIP: COSTAR_CVD_REVISED_RESULTS.zip')


All revised mathematics and decision-definition unit tests PASSED
Upload the original Cardiovascular_Disease_Dataset.csv file.


Saving Cardiovascular_Disease_Dataset.csv to Cardiovascular_Disease_Dataset.csv


COSTAR overall:   0%|                      | 0/19 [00:00 elapsed; ? ETA]

✓ Dataset schema, data audit and unstratified independent partitions
✓ Signed reweighting oracle verified against SciPy LP


primary: fully honest outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

...

without_cholesterol_variable: fully honest outer folds:   0%|          | 0/5 [00:00<?, ?it/s]

✓ Independent leakage ablation: without_cholesterol_variable
✓ Provenance manifest, interpretation report and final integrity checks
✓ High-resolution PNG, vector PDF, CSV and LaTeX exports finalized
FINAL ZIP: /content/COSTAR_CVD_REVISED_RESULTS.zip | 1.97 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Output folder: COSTAR_CVD_REVISED_RESULTS
FINAL downloadable ZIP: COSTAR_CVD_REVISED_RESULTS.zip


## Using the output

Return the downloaded `COSTAR_CVD_REVISED_RESULTS.zip` for a manuscript-ready Results, Verification, Discussion and Conclusion. In particular, the `README_RESULTS.md`, manually verified numerical tables and all vector-PDF plots support fully traceable reporting. **A tie, a failed certificate or a superior baseline is a valid scientific finding** and must not be concealed.